In [29]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

**1) Load dataset**

In [30]:
df = pd.read_csv(r"C:\Users\DELL\Downloads\output.csv")
df

,Review,Liked
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1
...,...,...
995,I think food should have flavor and texture an...,0
996,Appetite instantly gone.,0
997,Overall I was not impressed and would not go b...,0
998,"The whole experience was underwhelming, and I ...",0


**2) Check columns**

In [31]:
df.columns    # display column names

Index([' Review', 'Liked'], dtype='object')

**3) Display**
- First 10 rows

- Number of samples per class

- Missing values

- Review text length distribution


In [32]:
df.head(10)   # display first 10 rows

,Review,Liked
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1
5,Now I am getting angry and I want my damn pho.,0
6,Honeslty it didn't taste THAT fresh.),0
7,The potatoes were like rubber and you could te...,0
8,The fries were great too.,1
9,A great touch.,1


In [33]:
df['Liked'].value_counts()    # Number of samples per class

Liked
1    500
0    500
Name: count, dtype: int64

In [34]:
df.isnull().sum()   # Check missing values

 Review    0
Liked      0
dtype: int64

In [35]:
df["Review_length"] = df[" Review"].apply(len)
print(df["Review_length"].describe())

count    1000.000000
mean       58.315000
std        32.360052
min        11.000000
25%        33.000000
50%        51.000000
75%        80.000000
max       149.000000
Name: Review_length, dtype: float64


# TASK 2: Clean & Preprocess the Text
Perform the following text cleaning steps:
1. Convert all text to lowercase

2. Remove:

- punctuation

- numbers

- extra spaces

3. Remove stopwords (like the, is, are…)

4. Lemmatize or stem the words

5. Optional: remove emojis or special characters




**Import required libraries**

In [36]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [37]:
# convert all text to lower case
df[' Review'] = df[' Review'].str.lower()
df[' Review'].head()

0                             wow... loved this place.
1                                   crust is not good.
2            not tasty and the texture was just nasty.
3    stopped by during the late may bank holiday of...
4    the selection on the menu was great and so wer...
Name:  Review, dtype: object

In [38]:
# Remove punctuation, numbers, extra spaces
df[' Review'] = df[' Review'].apply(lambda x: re.sub(r'\s+', ' ', re.sub(r'[^a-zA-Z\s]', '', x)).strip())
df[' Review'].head()                                    

0                                 wow loved this place
1                                    crust is not good
2             not tasty and the texture was just nasty
3    stopped by during the late may bank holiday of...
4    the selection on the menu was great and so wer...
Name:  Review, dtype: object

In [39]:
# remove stop words
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

df[' Review'] = df[' Review'].apply(
    lambda x: " ".join(word for word in x.split() if word not in stop_words)
)

df[' Review'].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


0                                      wow loved place
1                                           crust good
2                                  tasty texture nasty
3    stopped late may bank holiday rick steve recom...
4                          selection menu great prices
Name:  Review, dtype: object

In [40]:
# lemmatizer
import nltk
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

df[' Review'] = df[' Review'].apply(
    lambda x: " ".join(
        lemmatizer.lemmatize(word)
        for word in x.split()
    )
)

df[' Review'].head()

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


0                                      wow loved place
1                                           crust good
2                                  tasty texture nasty
3    stopped late may bank holiday rick steve recom...
4                           selection menu great price
Name:  Review, dtype: object

# TASK 3: Convert Text to Numerical Features
Choose at least one technique:
1. Bag of Words (CountVectorizer)

2. TF-IDF Vectorizer

3. Compare both representations (optional task)


In [41]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()

x = cv.fit_transform(df[' Review'])
print(x.shape)

(1000, 1809)


In [42]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

x_tfidf = tfidf.fit_transform(df[' Review'])
print(x_tfidf.shape)

(1000, 1809)


Bag of Words converts text into numerical data by counting how many times each word appears in a review. It is simple and easy to implement, but very common words may get high importance even if they do not contribute much to the meaning.

TF-IDF (Term Frequency–Inverse Document Frequency) also converts text into numerical form, but it gives higher importance to meaningful words and reduces the importance of very common words. Because of this, TF-IDF usually provides better performance in sentiment analysis and text classification tasks.

For review datasets, TF-IDF is generally preferred because it captures important words more effectively than Bag of Words.

# TASK 4: Split the Data
Split dataset into:
- Training set → 80%

- Test set → 20%


In [43]:
from sklearn.model_selection import train_test_split

x = x
y = df['Liked']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [44]:
print("x_train shape:", x_train.shape)
print("x_test shape:",x_test.shape)

x_train shape: (800, 1809)
x_test shape: (200, 1809)


# TASK 5: Train Naïve Bayes Models
1. BernoulliNB


- Best for binary features or short reviews


In [45]:
from sklearn.naive_bayes import BernoulliNB

nb = BernoulliNB()

nb.fit(x_train, y_train)

y_pred = nb.predict(x_test)
y_pred

array([0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1,
       1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0,
       1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0,
       1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1,
       1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1,
       0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0,
       0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0,
       0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0,
       1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1,
       0, 1])

# TASK 6: Evaluate Models
For each NB model:
1. Calculate:

- Accuracy

- Precision

- Recall

- F1 Score


2. Show:

- Confusion matrix

- Classification report




In [46]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [47]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

precision = precision_score(y_test, y_pred)
print("Precision:", precision)

recall = recall_score(y_test, y_pred)
print("Recall:", recall)

f1 = f1_score(y_test, y_pred)
print("F1 Score:", f1)

cm = confusion_matrix(y_test, y_pred)
print(cm)

report = classification_report(y_test, y_pred)
print(report)

Accuracy: 0.75
Precision: 0.75
Recall: 0.7788461538461539
F1 Score: 0.7641509433962265
[[69 27]
 [23 81]]
              precision    recall  f1-score   support

           0       0.75      0.72      0.73        96
           1       0.75      0.78      0.76       104

    accuracy                           0.75       200
   macro avg       0.75      0.75      0.75       200
weighted avg       0.75      0.75      0.75       200



In [48]:
reviews = [
    "The food was fantastic!",
    "Worst service ever."
]

In [49]:
rev = tfidf.transform(reviews)
pred = nb.predict(rev)

for review, sentiment in zip(reviews, pred):
    if sentiment == 1:
        print(review," Positive Review")
    else:
        print(review," Negative Review")

The food was fantastic!  Positive Review
Worst service ever.  Negative Review


In [50]:
pip install streamlit

Note: you may need to restart the kernel to use updated packages.


In [51]:
import pickle
pickle.dump(tfidf, open("tfidf.pkl", "wb"))
pickle.dump(nb, open("model.pkl", "wb"))

In [52]:
import streamlit as st
import pickle
import warnings

warnings.filterwarnings("ignore")

# Load TF-IDF vectorizer and trained model
tfidf = pickle.load(open("tfidf.pkl", "rb"))
model = pickle.load(open("model.pkl", "rb"))

# Title
st.title("Restaurant Review Sentiment Analysis")

# User input
review = st.text_input("Enter your review")

# Prediction button
if st.button("Predict"):

    # Check empty input
    if review.strip() == "":
        st.warning("Please enter a review")

    else:
        # Transform review using TF-IDF
        review_vector = tfidf.transform([review])

        # Predict sentiment
        prediction = model.predict(review_vector)

        # Display result
        if prediction[0] == 1:
            st.success("Positive Review ")
        else:
            st.error("Negative Review ")

2026-05-15 17:39:39.947 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 17:39:39.949 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 17:39:39.951 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 17:39:39.953 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 17:39:39.955 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 17:39:39.957 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 17:39:39.958 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 17:39:39.960 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar